### AoC 2025 Day 10

In [1]:
lines = open('input10').readlines()
print('num lines:', len(lines))

num lines: 194


In [2]:
class Machine:
    def __init__(self, line: str) -> None:
        li: list[str] = line.split()
        self.lights = [x == '#' for x in list(li[0][1:-1])]
        self.buttons = list(map(lambda x: tuple(map(int, x[1:-1].split(','))), li[1:-1]))
        self.joltages = list(map(int, li[-1][1:-1].split(',')))
    def __repr__(self) -> str:
        return ("lights: " + repr(self.lights) + 
                "\nbuttons: " + repr(self.buttons) + 
                "\njoltages: " + repr(self.joltages))

In [3]:
machines = [Machine(s) for s in lines]
machines[0]

lights: [True, True, False, False, False, False, False, True, False]
buttons: [(0, 1, 5, 8), (1, 6, 7), (3, 6, 8), (1, 3, 6, 7), (0, 1, 2, 6, 7), (1, 2, 3, 5, 7), (0, 1, 3, 4, 5, 6, 7), (1, 2, 4, 5, 7, 8), (0, 2, 5, 7, 8), (1, 2, 3, 5, 7, 8)]
joltages: [53, 78, 43, 44, 33, 73, 46, 81, 60]

#### Part 1

In [4]:
import itertools
# Look at all subsets of buttons and try the subsets. 2^n subsets of set of size n.
def solve_subsets(lights_target: list, buttons: list) -> int:
    for subset_size in range(1, len(buttons) + 1):
        for buttons_to_push in itertools.combinations(buttons, subset_size):
            lights = [False] * len(lights_target)
            for button in buttons_to_push:
                for b in button:
                    lights[b] = not lights[b]
            if lights == lights_target:
                return subset_size
    return 0

print('Part 1:', sum(solve_subsets(m.lights, m.buttons) for m in machines))

Part 1: 558


#### Part 2

In [5]:
#!pip install cvxpy
import cvxpy as cp
import numpy as np

The problem can be formulated as an [integer linear program](https://en.wikipedia.org/wiki/Integer_programming):
$$
\begin{aligned}
\text{minimize} \quad & \mathbf{1}^\top \mathbf{x}  = \sum_{i=1}^{n}{x_i} \\
% \text{subject to} \quad & \mathbf{A}\mathbf{x} \leq \mathbf{b} \\
\text{subject to} \quad & \mathbf{M}\mathbf{x} = \mathbf{j} \\
& \mathbf{x} \geq \mathbf{0} \\
& \mathbf{x} \in \mathbb{Z}^n
\end{aligned}
$$

where $\mathbf{x} \in \mathbb{Z}^n$ is the (integer) number of presses for each button ($n$ buttons), $\mathbf{j} \in \mathbb{Z}^m$ is the joltage requirement, and $\mathbf{M} \in \{0,1\}^{m \times n}$ with $\mathbf{M}_{r,c} = 1$ if button $c$ affects joltage $r$. 

We use the `cvxpy` package to solve this optimization problem. 

In [6]:
def solve_cvx(machine: Machine):
    j = np.array(machine.joltages)
    M = np.zeros((len(j), len(machine.buttons)))
    for col in range(len(machine.buttons)): # unsparsify
        for row in machine.buttons[col]:
            M[row,col] = 1

    x = cp.Variable(len(machine.buttons), integer=True)
    objective = cp.sum(x)
    constraints = [M @ x == j, x >= 0]

    return cp.Problem(cp.Minimize(objective), constraints).solve()

In [7]:
print('Part 2:', round(sum(solve_cvx(m) for m in machines)))

Part 2: 20317


Could also use `z3`, `scipy.optimize` or an approach described [here on reddit](https://www.reddit.com/r/adventofcode/comments/1pk87hl/2025_day_10_part_2_bifurcate_your_way_to_victory/).